In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [4]:
# Load the saved feature dataset
df = pd.read_csv(r"C:\Users\moham\OneDrive\Desktop\KFUPM\Courses\242\AML\Project\ImageDataset\image_features.csv")

In [5]:
# Define 5 popular classifiers
classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "RandomForest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
}

In [6]:
from collections import defaultdict

# Store results
results = defaultdict(dict)

# Get unique CNN prefixes from column names
cnn_prefixes = set(col.split('_feature_')[0] for col in df.columns if '_feature_' in col)

# Encode class labels as integers
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["LabelEncoded"] = le.fit_transform(df["Label"])

# Loop over each CNN feature set
for cnn in sorted(cnn_prefixes):
    print(f"\n=== Using features from {cnn} ===")
    
    # Extract features for current CNN
    feature_cols = [col for col in df.columns if col.startswith(cnn)]
    X_all = df[feature_cols].values
    y_all = df["LabelEncoded"].values

    # Normalize features
    scaler = StandardScaler()
    X_all = scaler.fit_transform(X_all)

    # Manually stratified split
    X_train_list, X_test_list = [], []
    y_train_list, y_test_list = [], []

    for label in np.unique(y_all):
        idx = np.where(y_all == label)[0]
        np.random.shuffle(idx)  # Shuffle indices
        
        if len(idx) <= 10:
            # Very small class
            n_test = 1
        else:
            n_test = max(1, int(0.1 * len(idx)))

        test_idx = idx[:n_test]
        train_idx = idx[n_test:]

        X_train_list.append(X_all[train_idx])
        y_train_list.append(y_all[train_idx])
        X_test_list.append(X_all[test_idx])
        y_test_list.append(y_all[test_idx])

    # Final arrays
    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)
    X_test = np.vstack(X_test_list)
    y_test = np.concatenate(y_test_list)

    for clf_name, clf in classifiers.items():
        print(f"Training {clf_name}...")
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results[cnn][clf_name] = acc
        print(f"{clf_name} Accuracy on {cnn}: {acc:.4f}")



=== Using features from densenet121 ===
Training LogisticRegression...
LogisticRegression Accuracy on densenet121: 0.8839
Training SVM...
SVM Accuracy on densenet121: 0.8710
Training RandomForest...
RandomForest Accuracy on densenet121: 0.8903
Training KNN...
KNN Accuracy on densenet121: 0.8194

=== Using features from efficientnet_b0 ===
Training LogisticRegression...
LogisticRegression Accuracy on efficientnet_b0: 0.8903
Training SVM...
SVM Accuracy on efficientnet_b0: 0.8774
Training RandomForest...
RandomForest Accuracy on efficientnet_b0: 0.8839
Training KNN...
KNN Accuracy on efficientnet_b0: 0.8452

=== Using features from mobilenetv2 ===
Training LogisticRegression...
LogisticRegression Accuracy on mobilenetv2: 0.9097
Training SVM...
SVM Accuracy on mobilenetv2: 0.8645
Training RandomForest...
RandomForest Accuracy on mobilenetv2: 0.8774
Training KNN...
KNN Accuracy on mobilenetv2: 0.7871

=== Using features from resnet18 ===
Training LogisticRegression...
LogisticRegression A

In [10]:
# Convert results to DataFrame for easy viewing
results_df = pd.DataFrame(results).T  # Transpose so CNNs are rows, classifiers are columns
print("\n=== Summary of All Classifier Accuracies ===")
print(results_df)

    # Optionally save
results_df.to_csv("ml_classifier_results_per_cnn.csv")



=== Summary of All Classifier Accuracies ===
                 LogisticRegression       SVM  RandomForest       KNN
densenet121                0.883871  0.870968      0.890323  0.819355
efficientnet_b0            0.890323  0.877419      0.883871  0.845161
mobilenetv2                0.909677  0.864516      0.877419  0.787097
resnet18                   0.883871  0.858065      0.903226  0.819355
vgg16                      0.890323  0.877419      0.903226  0.845161


In [12]:
from sklearn.metrics import classification_report

# Loop over each CNN feature set
for cnn in sorted(cnn_prefixes):
    print(f"\n\n===== Best Model for CNN Features: {cnn} =====")
    
    # Extract features
    feature_cols = [col for col in df.columns if col.startswith(cnn)]
    X_all = df[feature_cols].values
    y_all = df["LabelEncoded"].values

    # Normalize
    scaler = StandardScaler()
    X_all = scaler.fit_transform(X_all)

    # Manual stratified split
    X_train_list, X_test_list = [], []
    y_train_list, y_test_list = [], []

    for label in np.unique(y_all):
        idx = np.where(y_all == label)[0]
        np.random.shuffle(idx)
        
        if len(idx) <= 5:
            n_test = 1
        else:
            n_test = max(1, int(0.1 * len(idx)))

        test_idx = idx[:n_test]
        train_idx = idx[n_test:]

        X_train_list.append(X_all[train_idx])
        y_train_list.append(y_all[train_idx])
        X_test_list.append(X_all[test_idx])
        y_test_list.append(y_all[test_idx])

    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)
    X_test = np.vstack(X_test_list)
    y_test = np.concatenate(y_test_list)

    # Find best classifier
    best_clf_name = max(results[cnn], key=results[cnn].get)
    best_clf = classifiers[best_clf_name]

    # Re-train best model
    best_clf.fit(X_train, y_train)
    y_pred = best_clf.predict(X_test)

    # Safely handle classes actually in test
    labels_in_test = np.unique(y_test)
    target_names_in_test = le.inverse_transform(labels_in_test)

    print(f"\nBest Classifier: {best_clf_name}")
    print(classification_report(
        y_test, y_pred,
        labels=labels_in_test,
        target_names=target_names_in_test
    ))




===== Best Model for CNN Features: densenet121 =====

Best Classifier: RandomForest
               precision    recall  f1-score   support

  Anserverbot       0.95      0.95      0.95        21
      Bmaster       1.00      1.00      1.00         1
   DroidDream       0.82      0.91      0.86        35
      Geinimi       0.88      0.96      0.92        24
      MisoSMS       1.00      0.89      0.94         9
     Nickyspy       0.88      0.79      0.83        19
NotCompatible       1.00      1.00      1.00         7
       Pletor       1.00      1.00      1.00         8
    Rootsmart       1.00      1.00      1.00         2
     Sandroid       1.00      0.25      0.40         4
     TigerBot       1.00      1.00      1.00         9
        Wroba       1.00      0.90      0.95        10
        Zitmo       0.86      1.00      0.92         6

     accuracy                           0.91       155
    macro avg       0.95      0.90      0.91       155
 weighted avg       0.92      0.

c:\Users\moham\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\moham\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\moham\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Best Classifier: LogisticRegression
               precision    recall  f1-score   support

  Anserverbot       1.00      1.00      1.00        21
      Bmaster       1.00      1.00      1.00         1
   DroidDream       0.84      0.91      0.88        35
      Geinimi       0.92      0.96      0.94        24
      MisoSMS       0.89      0.89      0.89         9
     Nickyspy       0.83      0.79      0.81        19
NotCompatible       1.00      1.00      1.00         7
       Pletor       1.00      1.00      1.00         8
    Rootsmart       1.00      1.00      1.00         2
     Sandroid       1.00      0.75      0.86         4
     TigerBot       1.00      1.00      1.00         9
        Wroba       1.00      0.80      0.89        10
        Zitmo       1.00      1.00      1.00         6

     accuracy                           0.92       155
    macro avg       0.96      0.93      0.94       155
 weighted avg       0.93      0.92      0.92       155



===== Best Model for CN

In [8]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# For consistent layout
import math

In [10]:
from sklearn.metrics import confusion_matrix

# Reset results storage
confusion_matrices = defaultdict(dict)

# Loop over each CNN feature set again
for cnn in sorted(cnn_prefixes):
    print(f"\n\n===== CNN Features: {cnn} =====")
    
    # Extract features
    feature_cols = [col for col in df.columns if col.startswith(cnn)]
    X = df[feature_cols].values
    y = df["LabelEncoded"].values

    # Normalize
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    for clf_name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        cm = confusion_matrix(y_test, y_pred)
        confusion_matrices[cnn][clf_name] = cm

        print(f"\n--- {clf_name} ---")
        print(cm)



===== CNN Features: densenet121 =====


c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--- LogisticRegression ---
[[41  0  1  0  0  0  0  0  0  1  1  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 1  0 64  1  0  3  0  0  0  2  0  1  0]
 [ 2  0  6 39  0  2  0  0  0  0  0  0  0]
 [ 0  0  1  0 17  0  0  0  0  0  2  0  0]
 [ 0  0  4  1  0 32  0  0  0  2  0  0  0]
 [ 0  0  0  0  0  0 15  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  5  0  0  0  0]
 [ 0  0  2  0  0  3  0  1  0  3  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 19  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0 13]]

--- SVM ---
[[41  0  2  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 1  0 67  2  1  1  0  0  0  0  0  0  0]
 [ 2  0  4 42  0  1  0  0  0  0  0  0  0]
 [ 2  0  0  2 15  1  0  0  0  0  0  0  0]
 [ 0  0  6  1  0 32  0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0 14  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  0  0  1  0  0  4  0  0  0  0]
 [ 0  0  4  0  0  0  0  0  0  5  0

c:\Users\moham\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--- LogisticRegression ---
[[41  0  1  0  0  2  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 64  2  2  3  0  0  0  1  0  0  0]
 [ 0  0  5 40  0  2  0  0  1  0  0  0  1]
 [ 0  0  0  0 20  0  0  0  0  0  0  0  0]
 [ 0  0  3  1  0 35  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 15  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  5  0  0  0  0]
 [ 0  0  3  0  0  2  0  0  0  3  0  0  1]
 [ 0  0  0  0  0  0  0  0  0  0 19  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0]
 [ 0  0  0  0  1  1  0  0  0  0  0  0 12]]

--- SVM ---
[[42  0  1  1  0  0  0  0  0  0  0  0  0]
 [ 0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 1  0 65  4  0  2  0  0  0  0  0  0  0]
 [ 2  0  3 43  0  1  0  0  0  0  0  0  0]
 [ 3  0  1  1 15  0  0  0  0  0  0  0  0]
 [ 0  0  3  3  0 33  0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0 14  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 17  0  0  0  0  0]
 [ 0  0  0  1  0  0  0  0  4  0  0  0  0]
 [ 0  0  4  1  0  1  0  0  0  3  0

KeyboardInterrupt: 